# Gemma 4 TP4 inference experiment
Run only after the fresh SDK 0.3.1 acceptance passes. Attach the same private offline datasets and licensed model. The three prompts are smoke checks, not software-engineering agent benchmarks. Request latency includes prefill and decode; reported tokens per second is not steady-state decode throughput.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess
base = Path('/kaggle/input/datasets/waqasm86')
sdk_input = base/'kaggle-vllm-sdk-v031-l4-acceptance'
wheel = sdk_input/'kaggle_vllm-0.3.1-py3-none-any.whl'
pip_wheel = sdk_input/'pip-25.2-py3-none-any.whl'
model = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
assert all(p.is_file() for p in (wheel,pip_wheel,model/'config.json'))
assert hashlib.sha256(wheel.read_bytes()).hexdigest() == 'a14a0be361ea0ab876aa939743ad7259ac012b8c9410cc668899da00b875c27b'
python312='/usr/bin/python3.12'
assert subprocess.run([python312,'-c','import sys;assert sys.version_info[:2]==(3,12)']).returncode == 0
env=dict(os.environ)
env['PYTHONPATH']=os.pathsep.join(filter(None,(str(pip_wheel),env.get('PYTHONPATH',''))))
env.update({'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','HF_DATASETS_OFFLINE':'1'})
sdk_target=Path('/tmp/kaggle-vllm-v031-experiment-sdk')
assert not sdk_target.exists(), 'Use a fresh Kaggle session'
subprocess.run([python312,'-m','pip','install','--no-index','--no-deps','--no-cache-dir','--target',str(sdk_target),str(wheel)],env=env,check=True,timeout=300)
env['PYTHONPATH']=os.pathsep.join((str(sdk_target),env['PYTHONPATH']))
bootstrap="from kaggle_vllm.l4_bootstrap import bootstrap; print(bootstrap(runtime_root='/kaggle/input/datasets/waqasm86', native_source='/kaggle/input/datasets/waqasm86/vllm-0-30-0-cp312-cp312-linux-x86-64/vllm-0.30.0-cp312-cp312-linux_x86_64.whl', strict=True))"
subprocess.run([python312,'-c',bootstrap],env=env,check=True,timeout=3600)
from pathlib import Path as _Path
runtime=_Path('/tmp/kaggle-vllm/v0.3.0/l4x4')
assert (runtime/'manifests/sdk-runtime.json').is_file()
env['PYTHONPATH']=os.pathsep.join((str(runtime/'pydeps'),env['PYTHONPATH'],str(sdk_input)))
env['LD_LIBRARY_PATH']=os.pathsep.join(filter(None,('/usr/local/nvidia/lib64',str(runtime/'pydeps/torch/lib'),env.get('LD_LIBRARY_PATH',''))))
env['LIBRARY_PATH']=os.pathsep.join(filter(None,(str(runtime/'cuda-driver-link'),env.get('LIBRARY_PATH',''))))
print('offline_runtime_ready',runtime)


In [ ]:
from pathlib import Path
import subprocess
script = "from pathlib import Path\nimport json, os, psutil, subprocess, threading, time\nfrom kaggle_vllm import KaggleLLM, KaggleSamplingParams\nfrom fresh_kaggle_gemma_v030 import child_snapshot, shutdown_and_verify\nassert os.environ.get('HF_HUB_OFFLINE') == '1'\nmodel=Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')\nparent=psutil.Process(); baseline=child_snapshot(parent,psutil)\nrows=[]; stop=threading.Event()\ndef sample_gpu():\n while not stop.is_set():\n  p=subprocess.run(['nvidia-smi','--query-gpu=memory.used,utilization.gpu','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=15)\n  if p.returncode==0:\n   try: rows.append([[int(v.strip()) for v in line.split(',')] for line in p.stdout.splitlines()])\n   except ValueError: pass\n  stop.wait(0.5)\nt=threading.Thread(target=sample_gpu,daemon=True);t.start()\nllm=None; result={'status':'FAIL','prompts':[],'model':str(model),'sdk_version':'0.3.1','native_version':'0.30.0','tp':4}\ntry:\n started=time.monotonic()\n llm=KaggleLLM(model=str(model),profile='kaggle-l4x4-cu129',tensor_parallel_size=4,dtype='bfloat16',gpu_memory_utilization=0.86,max_model_len=2048)\n result['load_seconds']=time.monotonic()-started\n result['attention_backend']=llm.effective_attention_backend()\n prompts=[('Explain tensor parallelism in one sentence.',None),('What is 2 + 3 + 5? Reply with the number only.','10'),('What is the length of the Python string CUDA? Reply with the number only.','4')]\n for prompt,expected in prompts:\n  start=time.monotonic()\n  output=llm.chat([{'role':'user','content':prompt}],local_files_only=True,sampling_params=KaggleSamplingParams(temperature=0.0,max_tokens=64))\n  elapsed=time.monotonic()-start\n  text=output[0].outputs[0].text.strip(); count=len(output[0].outputs[0].token_ids)\n  assert text and count>0\n  result['prompts'].append({'prompt':prompt,'text':text,'expected':expected,'exact_answer_pass':None if expected is None else text.strip().rstrip('.')==expected,'generated_tokens':count,'request_seconds':elapsed,'request_tokens_per_second':count/elapsed})\n result['status']='INFERENCE_COMPLETE_PENDING_CLEANUP'\nfinally:\n stop.set();t.join(timeout=20)\n result['gpu_samples']=len(rows)\n if rows:\n  result['peak_memory_mib_by_gpu']=[max(x[i][0] for x in rows if len(x)==4) for i in range(4)]\n  result['peak_sampled_utilization_percent_by_gpu']=[max(x[i][1] for x in rows if len(x)==4) for i in range(4)]\n if llm is not None:\n  cleanup=shutdown_and_verify(llm,parent,baseline,psutil)\n  result['cleanup']=cleanup['status']\n  result['status']='PASS' if result['status']=='INFERENCE_COMPLETE_PENDING_CLEANUP' and cleanup['status']=='PASS' else 'FAIL'\n Path('/kaggle/working/03_gemma4_tp4_inference.json').write_text(json.dumps(result,indent=2)+'\\n')\nprint(json.dumps({k:v for k,v in result.items() if k!='prompts'},indent=2))\nassert result['status']=='PASS'\n"
log=Path('/kaggle/working/03_gemma4_tp4_inference.log')
with log.open('w') as stream:
    result=subprocess.run([python312,'-c',script],env=env,stdout=stream,stderr=subprocess.STDOUT,timeout=2400)
print('exit',result.returncode,'log_tail','\n'.join(log.read_text(errors='replace').splitlines()[-30:]))
assert result.returncode==0
